# Full Analysis (Tuần 8) — MS · Nhóm 2

Mục 8.3 RBL-4 / proposal §5.6: metric + statistical test (α=0.05) + effect size + kết luận từng RQ → `summary.csv`.
**Notebook phải Restart & Run All không lỗi.**

> Sửa `INPUT` trỏ tới `results/full/metrics.csv`. Mặc định data giả để chạy thử.

In [ ]:
import os, csv, json, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats

INPUT = "data/metrics.sample.csv"   # <-- doi thanh results/full/metrics.csv
GPT, ALPHA, COV_THR, MUT_THR = "gpt4o-mini", 0.05, 80.0, 60.0
df = pd.read_csv(INPUT); df["compiled"]=df["compiled"].astype(int)
df.loc[df["compiled"]==0, ["branch_coverage","mutation_score"]] = 0.0
gpt = df[df["method"]==GPT].copy()
bname = next((m for m in df["method"].unique() if m!=GPT), None)
base = df[df["method"]==bname].copy()
print("GPT method:", GPT, "| baseline:", bname, "| N:", gpt["function_id"].nunique())

In [ ]:
# RQ1 - One-sample Wilcoxon: branch coverage GPT vs 80%
x = gpt["branch_coverage"].dropna().to_numpy(float); diff = x-COV_THR; nz = diff[diff!=0]
if nz.size:
    w,p = stats.wilcoxon(nz, alternative="greater")
else:
    w,p = None, None
rq1 = {"median": float(np.median(x)), "statistic": (float(w) if w is not None else None),
       "p_value": (float(p) if p is not None else None),
       "reject_H0": bool(p is not None and p<ALPHA and np.median(x)>=COV_THR)}
print("RQ1:", rq1)

In [ ]:
# RQ2 - Paired Wilcoxon: mutation GPT vs baseline + rank-biserial
m = pd.merge(gpt[["function_id","mutation_score"]], base[["function_id","mutation_score"]],
             on="function_id", suffixes=("_gpt","_base"))
g = m["mutation_score_gpt"].to_numpy(float); b = m["mutation_score_base"].to_numpy(float)
d = g-b; d2 = d[d!=0]
if d2.size:
    w2,p2 = stats.wilcoxon(g,b, alternative="greater")
    r = stats.rankdata(np.abs(d2)); rb = float((r[d2>0].sum()-r[d2<0].sum())/r.sum())
else:
    w2,p2,rb = None,None,0.0
rq2 = {"median_gpt": float(np.median(g)), "median_base": float(np.median(b)),
       "statistic": (float(w2) if w2 is not None else None),
       "p_value": (float(p2) if p2 is not None else None), "effect_size_rb": rb,
       "reject_H0": bool(p2 is not None and p2<ALPHA and np.median(g)>=MUT_THR)}
print("RQ2:", rq2)

In [ ]:
# RQ3 - Spearman: CC vs chat luong (GPT)
def sp(col):
    s = gpt[["cc",col]].dropna()
    if s["cc"].nunique()<2 or len(s)<3: return (None,None)
    rho,p = stats.spearmanr(s["cc"], s[col]); return (float(rho), float(p))
rho_c,p_c = sp("branch_coverage"); rho_m,p_m = sp("mutation_score")
rq3 = {"rho_cov":rho_c,"p_cov":p_c,"rho_mut":rho_m,"p_mut":p_m,
       "reject_H0": bool((rho_c is not None and rho_c<0 and p_c<ALPHA) or (rho_m is not None and rho_m<0 and p_m<ALPHA))}
print("RQ3:", rq3)

In [ ]:
# Ghi summary.csv (moi RQ 1 dong)
os.makedirs("results", exist_ok=True)
rows = [
 {"rq":"RQ1","metric":"branch_coverage","test":"one-sample Wilcoxon","statistic":rq1["statistic"],
  "p_value":rq1["p_value"],"effect_size":rq1["median"],"n":int(len(x)),
  "decision":"reject H0" if rq1["reject_H0"] else "fail to reject H0"},
 {"rq":"RQ2","metric":"mutation_score","test":"paired Wilcoxon","statistic":rq2["statistic"],
  "p_value":rq2["p_value"],"effect_size":rq2["effect_size_rb"],"n":int(len(m)),
  "decision":"reject H0" if rq2["reject_H0"] else "fail to reject H0"},
 {"rq":"RQ3","metric":"cc_vs_quality","test":"Spearman","statistic":None,
  "p_value":(min([v for v in [p_c,p_m] if v is not None], default=None)),
  "effect_size":f"rho_cov={rho_c}, rho_mut={rho_m}","n":int(gpt["cc"].notna().sum()),
  "decision":"reject H0" if rq3["reject_H0"] else "fail to reject H0"},
]
pd.DataFrame(rows).to_csv("results/summary.csv", index=False)
display(pd.DataFrame(rows))

In [ ]:
# Figures (>=300 DPI)
os.makedirs("results/figures", exist_ok=True)
fig,ax = plt.subplots(1,2, figsize=(10,4.5))
for a,col,t in [(ax[0],"branch_coverage","Branch Cov (GPT)"),(ax[1],"mutation_score","Mutation (GPT)")]:
    data=[gpt[gpt["language"]==L][col].dropna().to_numpy() for L in ["java","python"]]
    a.boxplot([d if len(d) else [0] for d in data], labels=["Java","Python"], showmeans=True)
    a.set_title(t); a.set_ylim(-5,105); a.grid(axis="y",alpha=.3)
fig.tight_layout(); fig.savefig("results/figures/fig1_distribution.png", dpi=300); plt.show()
fig,a = plt.subplots(figsize=(6,4.5))
a.boxplot([gpt["mutation_score"].dropna() if len(gpt) else [0], base["mutation_score"].dropna() if len(base) else [0]],
          labels=[f"GPT","Baseline"], showmeans=True)
a.set_title("Mutation: GPT vs Baseline"); a.set_ylim(-5,105); a.grid(axis="y",alpha=.3)
fig.tight_layout(); fig.savefig("results/figures/fig2_comparison.png", dpi=300); plt.show()

## Kết luận (diễn giải tổ hợp — proposal §6.2)
Đối chiếu `summary.csv`:
- **Double positive** (RQ1 & RQ2 reject H0): gpt-4o-mini đạt cả coverage lẫn mutation → thay được công cụ truyền thống ở CC 5–10.
- **Mixed** (RQ1 reject, RQ2 không): coverage cao nhưng mutation thấp → test "hình thức", thiếu assert phát hiện lỗi.
- **Double negative**: không đạt ngưỡng → one-shot không hiệu quả ở CC 5–10; chỉ ra ranh giới ứng dụng.
- RQ3 ρ<0 có ý nghĩa → CC càng cao chất lượng càng giảm (xác nhận "điểm gãy").

> Nhắc: kết luận nói rõ đối tượng là **gpt-4o-mini** (amendment v1.1). Bàn giao `summary.csv` + `figures/` cho RW.